# 02 — Data Cleaning & Standardization

Standardize potency values, deduplicate, handle censored values, and canonicalize SMILES for the raw KIT bioactivity pull from `01_data_collection.ipynb`.

See [Design Doc.md](../Design%20Doc.md) §5.1 and §4.4, and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 2.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW_PATH = Path("../data/raw/chembl_kit_bioactivity_raw.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print("Loaded:", df.shape)
df.head()

Loaded: (8703, 18)


,assay_chembl_id,assay_description,assay_type,canonical_smiles,document_chembl_id,document_year,molecule_chembl_id,pchembl_value,relation,standard_relation,standard_type,standard_units,standard_value,target_chembl_id,target_organism,type,units,value
0,CHEMBL820421,Inhibition of c-Kit autophosphorylation in int...,B,COc1cc2c(Oc3ccc(Nc4ccc(C(C)(C)C)cc4)cc3)ccnc2c...,CHEMBL1146677,2004.0,CHEMBL352308,7.00,=,=,IC50,nM,100.0,CHEMBL1936,Homo sapiens,IC50,nM,100.000
1,CHEMBL702237,Inhibition of KIT kinase activity,B,O=C(Cc1ccc2ccccc2c1)Nc1cc(C2CC2)n[nH]1,CHEMBL1148336,2004.0,CHEMBL115220,NaN,>,>,IC50,nM,10000.0,CHEMBL1936,Homo sapiens,IC50,nM,10000.000
2,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(OC(C)C)cc4)CC3)ncnc...,CHEMBL1135998,2002.0,CHEMBL330863,7.68,=,=,IC50,nM,21.0,CHEMBL1936,Homo sapiens,IC50,uM,0.021
3,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(OC(C)C)cc4)CC3)ncnc...,CHEMBL1135998,2002.0,CHEMBL124660,6.77,=,=,IC50,nM,170.0,CHEMBL1936,Homo sapiens,IC50,uM,0.170
4,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(C#N)cc4)CC3)ncnc2cc...,CHEMBL1135998,2002.0,CHEMBL126699,8.22,=,=,IC50,nM,6.0,CHEMBL1936,Homo sapiens,IC50,uM,0.006


## 1. Standardize potency values to pIC50/pKi/pEC50/pKd

Design Doc §5.1: convert to a common scale, pX = -log10(molar concentration), so IC50/Ki/Kd/EC50 values become comparable and roughly normally distributed. `standard_units` is almost all `nM` (8,549/8,703 — see `01_data_collection.ipynb` null counts), but the converter below handles the full range of molar-concentration unit strings ChEMBL uses, for robustness.

This step converts every row that *has* a value and unit, regardless of `standard_relation` (`=`, `>`, `<`, etc.) — a censored value like ">10000 nM" still converts to a valid bound, "pIC50 < 4.0". Deciding what to *do* with censored rows (drop/cap/flag) is a separate, later step, so the relation is carried through unchanged here rather than being resolved now.

In [2]:
# Molar-concentration unit strings ChEMBL uses -> multiplier to convert to molar (M).
UNITS_TO_MOLAR = {
    "M": 1,
    "mM": 1e-3,
    "uM": 1e-6,
    "nM": 1e-9,
    "pM": 1e-12,
    "fM": 1e-15,
}

unknown_units = set(df["standard_units"].dropna().unique()) - set(UNITS_TO_MOLAR)
assert not unknown_units, f"Unhandled units present, extend UNITS_TO_MOLAR: {unknown_units}"

molar = df["standard_value"] * df["standard_units"].map(UNITS_TO_MOLAR)
df["p_value"] = -np.log10(molar)

n_missing = df["p_value"].isna().sum()
print(f"p_value computed for {df['p_value'].notna().sum()} / {len(df)} rows")
print(f"{n_missing} rows have no p_value (missing standard_value and/or standard_units)")
df[["standard_type", "standard_relation", "standard_value", "standard_units", "pchembl_value", "p_value"]].head(10)

p_value computed for 8547 / 8703 rows
156 rows have no p_value (missing standard_value and/or standard_units)


,standard_type,standard_relation,standard_value,standard_units,pchembl_value,p_value
0,IC50,=,100.0,nM,7.00,7.000000
1,IC50,>,10000.0,nM,NaN,5.000000
2,IC50,=,21.0,nM,7.68,7.677781
3,IC50,=,170.0,nM,6.77,6.769551
4,IC50,=,6.0,nM,8.22,8.221849
5,IC50,=,4.0,nM,8.40,8.397940
6,IC50,=,260.0,nM,6.58,6.585027
7,IC50,=,190.0,nM,6.72,6.721246
8,IC50,=,60.0,nM,7.22,7.221849
9,IC50,=,16.0,nM,7.80,7.795880


### Cross-check against ChEMBL's own `pchembl_value`

ChEMBL independently computes `pchembl_value` (its own -log10 molar normalization) for a subset of records — mostly `=`-relation, non-censored ones. Our `p_value` should match it closely; large discrepancies would indicate a bug in the unit conversion above.

In [3]:
both = df.dropna(subset=["pchembl_value", "p_value"])
diff = (both["p_value"] - both["pchembl_value"]).abs()

print(f"{len(both)} rows have both p_value and pchembl_value")
print("Max abs difference:", diff.max())
print("Rows with |diff| > 0.01:", (diff > 0.01).sum())

assert diff.max() < 0.01, "p_value disagrees with ChEMBL's own pchembl_value — check unit conversion"

5711 rows have both p_value and pchembl_value
Max abs difference: 0.005483746814912038
Rows with |diff| > 0.01: 0
